# 02 · Choosing ε

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fairyflies9/fluidlets/blob/main/tutorials/02_choosing_eps.ipynb)

In [tutorial 01](01_blob_to_stokeslet.ipynb) a single regularized Stokeslet converged to the
singular one as $\varepsilon\to0$. So should we always take $\varepsilon$ as small as possible?
**No.** When a surface is represented by a finite number of points, a second error appears that
grows as $\varepsilon$ shrinks. This notebook measures both errors on a problem with an exact
answer, a rotating sphere, and turns the result into a rule for picking $\varepsilon$.

We reproduce the trend of Table II in Cortez, Fauci & Medovikov, Phys. Fluids **17**, 031504
(2005), Sec. IV.

**You will learn**

* how to set up and solve a *resistance problem* with `fluidlets.resistance`,
* what the regularization error and the discretization error are, and why they pull in opposite directions,
* how to choose $\varepsilon$ relative to the point spacing $h$.

In [ ]:
# On Colab or Binder, install Fluidlets first (skipped when it is already installed).
import importlib.util, subprocess, sys
if importlib.util.find_spec("fluidlets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/fairyflies9/fluidlets"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fluidlets as fl
from fluidlets.geometry import cube_sphere

kernel = fl.Stokeslet3D()
mu, a = 1.0, 1.0                     # viscosity and sphere radius
ez = np.array([0.0, 0.0, 1.0])

## 1. The test problem: a rotating sphere

A sphere of radius $a$ rotating with angular velocity $\boldsymbol\Omega$ in still fluid feels a
torque $\mathbf L = -8\pi\mu a^3\boldsymbol\Omega$. The torque the sphere exerts on the fluid is
$+8\pi\mu a^3\boldsymbol\Omega$, so for $\mu = a = 1$ and $\boldsymbol\Omega = \mathbf e_z$ the
resistance coefficient is

$$
R_{33} = 8\pi \approx 25.133 .
$$

**Discretization.** `cube_sphere(n)` puts an $n\times n$ grid on each face of a cube and projects
it onto the sphere, giving $6n^2$ points with spacing $h \approx \pi a/(2n)$ (a quarter of a great
circle per face edge).

In [ ]:
X = cube_sphere(6, a)
fig = plt.figure(figsize=(4.5, 4.5))
ax = fig.add_subplot(projection="3d")
ax.scatter(*X.T, s=8)
ax.set_box_aspect((1, 1, 1)); ax.set_title(f"cube_sphere(6): {len(X)} points")
plt.show()

**The resistance problem.** Every point $\mathbf x_k$ is told to move rigidly,
$\mathbf u(\mathbf x_k) = \boldsymbol\Omega\times\mathbf x_k$, and we solve for the forces
$\mathbf f_j$ on the fluid that make it happen:

$$
\sum_{j=1}^{N} \big[H_1(r_{kj})\,\mathbf I + H_2(r_{kj})\,\mathbf d_{kj}\mathbf d_{kj}^T\big]\,\mathbf f_j = \boldsymbol\Omega\times\mathbf x_k,
\qquad k = 1,\dots,N .
$$

That is a dense $3N\times3N$ linear system. `fl.resistance(kernel, points, velocities, eps, mu)`
assembles and solves it and returns the forces (shape `(N, 3)`) plus some solver information. The
torque is then $\sum_j \mathbf x_j\times\mathbf f_j$.

In [ ]:
def R33(n, eps):
    """Torque on the fluid from a unit sphere (6 n^2 points) rotating about z."""
    X = cube_sphere(n, a)
    forces, info = fl.resistance(kernel, X, np.cross(ez, X), eps, mu, method="lu")
    return np.cross(X, forces)[:, 2].sum(), forces, info

value, forces, info = R33(12, 0.05)
print(f"R33 = {value:.4f}   (exact 8 pi = {8 * np.pi:.4f}),  solver: {info}")

## 2. Reproducing Table II of Cortez, Fauci & Medovikov (2005)

The paper lists $R_{33}$ for three blob widths and four grids. At $n = 12$ ($6\times12\times12 = 864$
points) it reports 27.09, 25.53 and 19.62 for $\varepsilon = 0.1, 0.05, 0.01$.

In [ ]:
# CFM 2005, Table II: R33 for (eps, n)
paper = {(0.1, 12): 27.09, (0.1, 24): 27.16, (0.1, 36): 27.16, (0.1, 48): 27.16,
         (0.05, 12): 25.53, (0.05, 24): 26.08, (0.05, 36): 26.10, (0.05, 48): 26.11,
         (0.01, 12): 19.62, (0.01, 24): 23.89, (0.01, 36): 24.80, (0.01, 48): 25.09}

print("  eps    R33 (ours)   R33 (paper)")
for eps in [0.1, 0.05, 0.01]:
    ours = R33(12, eps)[0]
    print(f"{eps:5.2f}  {ours:10.2f}  {paper[(eps, 12)]:10.2f}")
    assert abs(ours - paper[(eps, 12)]) < 0.01

They agree to the digits printed. Two things already stand out:

* the **smallest** $\varepsilon$ gives the **worst** answer (19.62 vs 25.13), and
* $\varepsilon = 0.1$ overshoots while $\varepsilon = 0.01$ undershoots. The error changes sign.

## 3. A sweep over ε and point spacing

Now vary both $\varepsilon$ and $n$. Each solve takes about a second, so this cell runs for 10–20 s.

In [ ]:
ns = np.array([6, 9, 12, 15, 18])
epsilons = np.array([0.2, 0.1, 0.05, 0.03, 0.02, 0.01])
h = np.pi * a / (2 * ns)                        # point spacing for each n

table = np.array([[R33(n, eps)[0] for eps in epsilons] for n in ns])
rel_err = table / (8 * np.pi) - 1

print("relative error R33 / (8 pi) - 1")
print("   n      h  " + "".join(f"  eps={e:<6}" for e in epsilons))
for n, hn, row in zip(ns, h, rel_err):
    print(f"{n:4d} {hn:6.3f}  " + "".join(f"  {v:+9.4f}" for v in row))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(epsilons)))
for j, (eps, c) in enumerate(zip(epsilons, colors)):
    axs[0].plot(ns, table[:, j], "o-", color=c, label=f"eps = {eps}")
    pn = [n for n in (12, 24, 36, 48) if (eps, n) in paper]
    axs[0].plot(pn, [paper[(eps, n)] for n in pn], "s", mfc="none", color=c, ms=8)
axs[0].axhline(8 * np.pi, color="k", ls="--", label=r"exact $8\pi$")
axs[0].set_xlabel("n  (6 n² points)"); axs[0].set_ylabel("$R_{33}$")
axs[0].set_title("open squares: CFM 2005 Table II"); axs[0].legend(fontsize=8)

for i, n in enumerate(ns):
    axs[1].semilogx(epsilons, rel_err[i], "o-", label=f"n = {n}, h = {h[i]:.3f}")
axs[1].axhline(0, color="k", lw=0.8)
axs[1].set_xlabel(r"$\varepsilon$"); axs[1].set_ylabel("relative error")
axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Two competing errors

Read the left panel along each curve (fixed $\varepsilon$, more points) and the right panel along
each curve (fixed $h$, smaller $\varepsilon$).

**Regularization error, $O(\varepsilon)$.** With enough points every curve on the left levels off,
but not at $8\pi$: at $\varepsilon = 0.2, 0.1, 0.05$ the plateaus are about 17 %, 8 % and 4 % too
high. Halving $\varepsilon$ halves the error, so it is first order. Even with exact quadrature we are
solving for a sphere whose surface forces are smeared over a shell of thickness $\sim\varepsilon$;
the blobs push fluid a little outside the true surface, the body acts slightly larger, and the
torque is too big. (A single Stokeslet converged like $\varepsilon^2$ in tutorial 01, but a *surface*
of forces only gives $O(\varepsilon)$ near the surface; see CFM 2005, Sec. III.)

**Discretization (quadrature) error, grows with $h/\varepsilon$.** The sum over points approximates
an integral over the surface of a kernel that varies on the length scale $\varepsilon$. When the
spacing $h$ is much larger than $\varepsilon$ the blobs no longer overlap: the surface becomes a
sieve, fluid slips between the points, and the torque is too *small*. This error disappears as
$h/\varepsilon\to0$ and blows up as $\varepsilon\to0$ at fixed $h$, the opposite of the
regularization error.

We can see the leak directly. Along the $x$-axis the line leaves the sphere through the centre of a
face, which is a gap between four grid points. Inside, the fluid should rotate rigidly,
$u_y = x$; outside, it follows the rotlet $u_y = a^3/x^2$.

In [ ]:
X = cube_sphere(12, a)
line = np.c_[np.linspace(0, 2, 401), np.zeros(401), np.zeros(401)]
exact = np.where(line[:, 0] < a, line[:, 0], a**3 / np.maximum(line[:, 0], a) ** 2)

fig, ax = plt.subplots(figsize=(6, 3.8))
for eps in [0.1, 0.05, 0.01]:
    forces = R33(12, eps)[1]
    u = fl.evaluate(kernel, line, X, forces, eps, mu)
    ax.plot(line[:, 0], u[:, 1], label=f"eps = {eps}")
ax.plot(line[:, 0], exact, "k--", label="exact")
ax.axvline(a, color="gray", lw=0.5)
ax.set_xlabel("x"); ax.set_ylabel("$u_y$"); ax.set_title(f"n = 12, h = {np.pi / 24:.3f}")
ax.legend(); plt.show()

With $\varepsilon = 0.01 \ll h$ the fluid in the gap reaches only about 70 % of the wall speed:
it slips between the points, the interior lags behind rigid rotation, and the flow outside (whose
strength is set by the torque) is too weak. With $\varepsilon = 0.1$ the surface is watertight, but
the flow is smeared across $|x - a|\lesssim\varepsilon$ and is a little too strong outside, as if
the sphere were slightly bigger. $\varepsilon = 0.05$ is close to the exact curve everywhere except
right at the surface.

## 5. A rule for ε

The total error is roughly $C_1\,\varepsilon - C_2\,g(h/\varepsilon)$ with $g$ increasing. For each
$h$ there is a crossing $\varepsilon^*$ where the two cancel. Let us locate it by interpolating
each row of the sweep (in $\log\varepsilon$) and compare it with $h$.

In [ ]:
eps_star = []
for row in rel_err:
    k = np.nonzero(np.diff(np.sign(row)))[0][0]        # first sign change, from + to -
    t = row[k] / (row[k] - row[k + 1])
    eps_star.append(np.exp(np.log(epsilons[k]) + t * np.log(epsilons[k + 1] / epsilons[k])))
eps_star = np.array(eps_star)

print("   n      h     eps*   eps*/h")
for n, hn, es in zip(ns, h, eps_star):
    print(f"{n:4d} {hn:6.3f} {es:7.4f} {es / hn:7.3f}")

The crossing sits at $\varepsilon^*\approx h/3$ for every grid. That is the practical lesson:

> **Choose $\varepsilon$ proportional to the point spacing, $\varepsilon = c\,h$, never independently of it.**
> Then refining the surface also shrinks $\varepsilon$, both errors fall together, and the overall
> error is $O(h)$.

Some caveats worth remembering:

* The constant $c$ depends on the geometry, the point layout and the quantity you care about. Here
  $c\approx 1/3$ makes the *torque* right; the flow right next to the surface, or the force on a
  slender filament, may prefer a different $c$. Ainley et al. (2008), in tutorial 03, use
  $\varepsilon\propto h^{0.9}$ with different constants for different directions of motion.
* An exact cancellation is luck, not accuracy. The safe statement is that for $\varepsilon$ of order
  $h$ both errors are of order $h$.
* For a new problem, do what we did here: pick a quantity you can check (or refine until it stops
  changing) and sweep $\varepsilon/h$.

## Summary

* Resistance problem: prescribe velocities at points, solve `fl.resistance` for the forces, sum
  them (or their moments) to get the drag (or torque).
* Regularization error: $O(\varepsilon)$ for forces spread over a surface, pushes the drag up.
* Discretization error: grows with $h/\varepsilon$, makes the surface leaky, pulls the drag down.
* Rule of thumb: $\varepsilon = c\,h$ with $c$ of order one (here $c\approx1/3$); check it on your problem.

## Exercises

1. **Translation.** Repeat the sweep for a sphere translating with $\mathbf U = \mathbf e_z$
   (velocities `np.tile(ez, (len(X), 1))`) and compare the total force with $6\pi\mu a U$. Is the
   best $\varepsilon/h$ the same as for rotation?
2. **ε tied to h.** For $c = 1/4, 1/3, 1/2, 1$, set $\varepsilon = c\,h$ and plot the error in $R_{33}$
   against $h$ for $n = 6,\dots,18$ on log-log axes. What order of convergence do you see? Does it
   depend on $c$?
3. **Point layout.** Use `fluidlets.geometry.fibonacci_sphere(N)` (with spacing $h\approx\sqrt{4\pi a^2/N}$)
   instead of `cube_sphere`. Does the crossing $\varepsilon^*/h$ move?
4. **Conditioning.** Compute `np.linalg.cond(fl.assemble(kernel, X, X, eps))` for $n = 9$ and
   $\varepsilon = 0.2, 0.05, 0.01$. Then solve with `method="gmres"` and compare the iteration counts
   in `info`. Which $\varepsilon$ makes the linear system hard, and why (think about overlapping blobs)?
5. **Interior flow.** For $n = 12$, evaluate the velocity at 200 random points with $|\mathbf x| < 0.8$
   and plot the maximum deviation from rigid rotation against $\varepsilon$. Is the best $\varepsilon$
   for the interior flow the same as the best $\varepsilon$ for the torque?